# Hands-On 1: Variational AutoEncoder on 2-D Ising Model Configurations

The 2D Ising model on an $L \times L$ lattice has Hamiltonian
$$H = -J \sum_{\langle i,j \rangle} s_i s_j$$
where $s_i \in \{-1, +1\}$ and the sum runs over nearest-neighbor pairs.
The exact critical temperature (Onsager) is
$$T_c = \frac{2J}{k_B \ln(1+\sqrt{2})} \approx 2.269 \; J/k_B.$$


## Plan

A well-trained VAE with a 2D latent space should spontaneously organize configurations
by phase — without ever seeing the temperature label.

- Generate synthetic 2-D Ising spin configurations using Markov Chain Monte Carlo
- Train a convolutional VAE with a 2-D latent space to learn the ordered/disordered phase, without peeking at the temperature.


### GPU setup

Training generative models can take a while. It is good to run this on a GPU if you have one.

Zip ahead and start generate the Ising datasets now--it will take about 10-15 minutes with a Colab T4 TPU.

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

DEVICE

In [ ]:
# Pull in all packages at once so that we catch any problems at the beginning
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from tqdm.auto import tqdm

## Generate 2-D Ising model configurations

We generate spin configurations at a range of temperatures using the
Metropolis algorithm. Each configuration is an $L \times L$ matrix of $\pm 1$ spins.

I picked $L = 32$ and $N_{\text{configs}} = 200$ per temperature
to keep data generation time around 10 minutes in Colab.

In [ ]:
def metropolis_ising(L=32, T=2.0, n_steps=50_000, J=1.0):
    """
    Run Metropolis-Hastings to generate spins on an L x L Ising model
     at a specific temperature T.
    Returns the final spin configuration as an (L, L) array of +/-1.

    Parameters
    ----------
    L       : lattice side length
    T       : temperature in units of J/k_B
    n_steps : number of single-spin flip proposals
    J       : coupling constant in Hamiltonian (set to 1)
    """
    beta = 1.0 / T
    # Random initial configuration (instead of ones or zeroes)
    spins = np.random.choice([-1, 1], size=(L, L))

    for _ in range(n_steps):
        # Pick a random site
        i = np.random.randint(0, L)
        j = np.random.randint(0, L)
        # Compute energy change with periodic boundary conditions
        neighbors = (
            spins[(i+1) % L, j] +
            spins[(i-1) % L, j] +
            spins[i, (j+1) % L] +
            spins[i, (j-1) % L]
        )
        delta_E = 2.0 * J * spins[i, j] * neighbors
        # Metropolis acceptance criterion
        #  Depends on the change in energy relative to the temperature
        if delta_E < 0 or np.random.rand() < np.exp(-beta * delta_E):
            spins[i, j] *= -1

    # The output of this function is the LxL spin configuration
    return spins.astype(np.float32)


def generate_dataset(L=32, temperatures=None, n_configs=200, n_steps=50_000):
    """
    Generate Ising configurations across a range of temperatures.

    Returns
    -------
    configs : (N, 1, L, L) float32 array, values in {0, 1} (rescaled from {-1,+1})
    temps   : (N,) float32 array of temperatures
    """
    if temperatures is None:
        temperatures = np.linspace(1.0, 3.5, 11)  # 11 temperature steps

    configs_list = []
    temps_list = []

    for T in tqdm(temperatures, desc='Generating configs'):
        for _ in range(n_configs):
            spins = metropolis_ising(L=L, T=T, n_steps=n_steps)
            # Rescale from {-1,+1} to {0,1} for binary cross-entropy
            configs_list.append((spins + 1.0) / 2.0)
            temps_list.append(T)

    configs = np.stack(configs_list)[:, np.newaxis, :, :]  # (N, 1, L, L)
    temps = np.array(temps_list, dtype=np.float32)
    return configs, temps


print('Generating Ising dataset ...')
L = 32
T_CRIT = 2.0 / np.log(1.0 + np.sqrt(2.0))  # approx 2.269
temperatures = np.linspace(1.0, 3.5, 11)
configs, temps = generate_dataset(L=L, temperatures=temperatures,
                                   n_configs=150, n_steps=40_000)
print(f'Dataset shape: {configs.shape}, temperature range: [{temps.min():.2f}, {temps.max():.2f}]')

### Visualizing spin configurations

We see the progression from highly-ordered to disordered as the temperature increases.

In [ ]:
# Visualize sample configurations at different temperatures
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
sample_temps = [1.0, 1.8, 2.27, 2.8, 3.5]
for ax, t in zip(axes, sample_temps):
    idx = np.argmin(np.abs(temps - t))
    ax.imshow(configs[idx, 0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'T = {t:.2f}' + (' ≈ T_c' if abs(t - T_CRIT) < 0.1 else ''),
                 fontsize=10)
    ax.axis('off')
plt.suptitle('2-D Ising configurations at selected temperatures', y=1.02)
plt.tight_layout()
plt.show()

### Setting up the data loaders

Best practice, as usual, is to wrap the dataset in a class wrapper.

The `temps` are not used during training, but we keep them together with the `configs` so that we can use them later for the interpretation.

In [ ]:
class IsingDataset(Dataset):
    def __init__(self, configs, temps):
        self.configs = torch.tensor(configs, dtype=torch.float32)
        self.temps   = torch.tensor(temps,   dtype=torch.float32)

    def __len__(self):
        return len(self.configs)

    def __getitem__(self, idx):
        return self.configs[idx], self.temps[idx]


dataset = IsingDataset(configs, temps)
n_train = int(0.85 * len(dataset))
n_val   = len(dataset) - n_train
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

## Define VAE architecture

The VAE is an encoder-decoder structure, with `latent_dim` of 2 dimensions to allow direct visualization.

The "convolutional" part comes in processing of the 2-D Ising model "image" in early encoding layers.

The ELBO loss is:
$$L = -\langle[\log p_\theta(x|z)\rangle
+ D_{\text{KL}}(q_\phi(z|x) \| p(z))$$

For a Gaussian encoder $q_\phi$ and standard normal prior $p(z)$,
the KL term has the closed form:
$$D_{\text{KL}} = -\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

In [ ]:
class Encoder(nn.Module):
# Map (1, L=32, L=32) spin config to (mu, log_var) in R^latent_dim.
# mu, log_var define the probability distributions in R^latent_dim.

    def __init__(self, latent_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),  # -> (16, 16, 16)
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), # -> (32, 8, 8)
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, stride=2, padding=1), # -> (32, 4, 4)
            nn.ReLU(),
            nn.Flatten(),                               # -> 512
            nn.Linear(512, 128),
            nn.ReLU(),
        )
        self.fc_mu      = nn.Linear(128, latent_dim)
        self.fc_log_var = nn.Linear(128, latent_dim)

    def forward(self, x):
        h = self.net(x)
        return self.fc_mu(h), self.fc_log_var(h)


class Decoder(nn.Module):
    """Maps z in R^latent_dim back to (1, 32, 32) logits."""

    def __init__(self, latent_dim=2):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 512),
            nn.ReLU(),
        )
        self.deconv = nn.Sequential(
            nn.Unflatten(1, (32, 4, 4)),                        # -> (32, 4, 4)
            nn.ConvTranspose2d(32, 32, 3, stride=2,             # -> (32, 8, 8)
                               padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2,             # -> (16, 16, 16)
                               padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2,              # -> (1, 32, 32)
                               padding=1, output_padding=1),
            # No sigmoid here — we use BCEWithLogitsLoss
        )

    def forward(self, z):
        return self.deconv(self.fc(z))


class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
        self.latent_dim = latent_dim

    def reparameterize(self, mu, log_var):
        """Sample z = mu + eps * sigma using the reparameterization trick."""
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = self.reparameterize(mu, log_var)
        x_logits = self.decoder(z)
        return x_logits, mu, log_var

    def elbo_loss(self, x, beta=1.0):
        """
        Compute the beta-ELBO loss.

        Parameters
        ----------
        x    : input batch (B, 1, L, L), values in [0, 1]
        beta : weight on the KL term (beta=1 is the standard ELBO)

        Returns
        -------
        loss      : scalar total loss
        recon_loss: reconstruction term
        kl_loss   : KL divergence term
        """
        x_logits, mu, log_var = self(x)
        # Reconstruction: binary cross-entropy per pixel, summed over pixels
        recon_loss = F.binary_cross_entropy_with_logits(
            x_logits, x, reduction='sum') / x.size(0)
        # KL divergence: closed form for Gaussian encoder vs N(0,I) prior
        kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)
        return recon_loss + beta * kl_loss, recon_loss, kl_loss


model = VAE(latent_dim=2).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'VAE parameters: {n_params:,}')

## Training the VAE

This is MUCH faster than generating the synthetic data! Part of the reason is that we constructed a VAE model that is simple to train.

In [ ]:
def train_vae(model, train_loader, val_loader, n_epochs=30, lr=3e-4, beta=1.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    history = {'train_loss': [], 'val_loss': [], 'train_kl': [], 'val_kl': []}

    for epoch in tqdm(range(n_epochs), desc='Training'):
        # ── Training ──
        model.train()
        train_loss_sum = train_kl_sum = 0.0
        for x_batch, _ in train_loader:
            x_batch = x_batch.to(DEVICE)
            optimizer.zero_grad()
            loss, recon, kl = model.elbo_loss(x_batch, beta=beta)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item()
            train_kl_sum   += kl.item()

        # ── Validation ──
        model.eval()
        val_loss_sum = val_kl_sum = 0.0
        with torch.no_grad():
            for x_batch, _ in val_loader:
                x_batch = x_batch.to(DEVICE)
                loss, recon, kl = model.elbo_loss(x_batch, beta=beta)
                val_loss_sum += loss.item()
                val_kl_sum   += kl.item()

        scheduler.step()

        history['train_loss'].append(train_loss_sum / len(train_loader))
        history['val_loss'].append(val_loss_sum   / len(val_loader))
        history['train_kl'].append(train_kl_sum   / len(train_loader))
        history['val_kl'].append(val_kl_sum       / len(val_loader))

    return history


history = train_vae(model, train_loader, val_loader, n_epochs=30, lr=3e-4, beta=1.0)

### Visualize training history



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('ELBO loss')
axes[0].set_title('Total loss')
axes[0].legend()

axes[1].plot(history['train_kl'], label='Train')
axes[1].plot(history['val_kl'],   label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('KL divergence')
axes[1].set_title('KL term')
axes[1].legend()

plt.tight_layout()
plt.show()

## Interpretation: Latent space and Phase Transition

We can peek directly in the latent space to see how the VAE has encoded the different configurations. Remember that this is the only information the decoder will get from the configuration.

We can see that the disordered population clusters near 0 in the latent space, while the ordered population has a wider spread.

In [ ]:
# Encode the full dataset and collect latent coordinates
model.eval()
all_mu = []
all_temps = []

full_loader = DataLoader(dataset, batch_size=256, shuffle=False)
with torch.no_grad():
    for x_batch, t_batch in full_loader:
        mu, _ = model.encoder(x_batch.to(DEVICE))
        all_mu.append(mu.cpu().numpy())
        all_temps.append(t_batch.numpy())

all_mu    = np.concatenate(all_mu,    axis=0)
all_temps = np.concatenate(all_temps, axis=0)

# Plot latent space colored by temperature
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(all_mu[:, 0], all_mu[:, 1],
                c=all_temps, cmap='coolwarm', s=8, alpha=0.7)
plt.colorbar(sc, ax=ax, label='Temperature T')
ax.axvline(0, color='k', lw=0.5, ls='--')
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_xlabel('z_1')
ax.set_ylabel('z_2')
ax.set_title('VAE latent space colored by temperature\n(blue = ordered, red = disordered)')
plt.tight_layout()
plt.show()

### Interpretation: latent space and magnetization

Our hypothesis is that the latent space should include the information about magnetization, indicating that the VAE learned about order/disorder through that variable.

Hmm, it doesn't look so good for $z_1$. Try plotting $m$ vs. $z_2$ instead.

In [ ]:
# Magnetization as a function of latent coordinate z_1
# Expectation: z_1 (or z_2) should correlate strongly with |m|

# Compute magnetization for each config (rescale back to {-1,+1} first)
raw_configs = configs[:, 0, :, :]  # (N, L, L)
spins_pm1   = 2.0 * raw_configs - 1.0
magnetization = np.abs(spins_pm1.mean(axis=(1, 2)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc1 = axes[0].scatter(all_mu[:, 0], magnetization,
                      c=all_temps, cmap='coolwarm', s=8, alpha=0.6)
plt.colorbar(sc1, ax=axes[0], label='Temperature')
axes[0].set_xlabel('z_1 (latent dimension 1)')
axes[0].set_ylabel('|m| (magnetization)')
axes[0].set_title('Magnetization vs z_1')

# Mean magnetization per temperature — compare to analytic Onsager result
T_vals = np.sort(np.unique(all_temps))
mean_mag = [magnetization[all_temps == T].mean() for T in T_vals]
axes[1].plot(T_vals, mean_mag, 'o-', label='Measured |m|')
axes[1].axvline(T_CRIT, color='r', ls='--', label=f'T_c = {T_CRIT:.3f}')
axes[1].set_xlabel('Temperature T')
axes[1].set_ylabel('Mean |m|')
axes[1].set_title('Order parameter vs temperature')
axes[1].legend()

plt.tight_layout()
plt.show()

## Discussion and experiments with the 2-D Ising VAE

- Does the VAE latent space separate the two phases without supervision?
   What physical quantity does each latent dimension correspond to?
- What happens to the latent space if you increase `beta` to 4 or 8 (beta-VAE)? I.e., does the latent space become more or less disentangled?
- Train a plain autoencoder (remove the KL term, i.e., set `beta=0`).
   How does the latent space structure change? Can you still interpolate smoothly?


# Hands-On 2: Diffusion Model to generate Cherenkov telescope waveforms

Imaging Atmospheric Cherenkov Telescopes (IACTs) record brief flashes of
Cherenkov light from particle showers. Each camera pixel in the CT produces a
time-resolved waveform: a short 1-D time series of photoelectron counts.

- Gamma-ray showers produce narrow, symmetric waveforms (electromagnetic cascade)
- Cosmic-ray (hadronic) showers produce broader, irregular waveforms
  with sub-pulses from secondary pions

We will simulate waveforms analytically and train a conditional diffusion
model to generate them.

## Plan

Diffusion model will generate waveforms more quickly than other generative models.

- Simulate realistic Cherenkov telescope waveforms for gamma-ray and cosmic-ray showers
- Train a 1-D diffusion model (DDPM) conditioned on shower type

## Generate synthetic data waveforms in 2 classes

We could have drawn from some real ICAT data or from a physics-motivated simulation.
In the interest of time, we will create synthetic data and pretend this is the target for our diffusion model.

### Gamma waveform
Single narrow Gaussian pulse:
$$w(t) = A \exp\!\left(-\frac{(t - t_0)^2}{2\sigma^2}\right) + \epsilon(t)$$

### Hadron waveform
Superposition of the primary pulse plus 1–3 pion sub-shower
pulses at random delays times :
$$w(t) = \sum_{k} A_k \exp\!\left(-\frac{(t - t_k)^2}{2\sigma_k^2}\right) + \epsilon(t)$$

We can add readout noise $\epsilon(t)$, drawing from a Poisson distribution.

Try running with `noise_level=0` first, so that you can see the waveforms better without noise. Then increase the noise to something like `noise_level=0.05`.

In [ ]:
N_SAMPLES = 64    # time samples per waveform
T_AXIS    = np.linspace(0, 1, N_SAMPLES, dtype=np.float32)


def gaussian_pulse(t, amplitude, center, width):
    return amplitude * np.exp(-0.5 * ((t - center) / width) ** 2)


def simulate_gamma(t=T_AXIS, noise_level=0.05):
    #Single narrow Gaussian pulse: electromagnetic cascade.
    center    = np.random.uniform(0.3, 0.7)
    amplitude = np.random.uniform(0.7, 1.0)
    width     = np.random.uniform(0.04, 0.08)  # narrow
    waveform  = gaussian_pulse(t, amplitude, center, width)
    waveform += np.random.normal(0, noise_level, size=len(t)).astype(np.float32)
    return waveform.astype(np.float32)


def simulate_hadron(t=T_AXIS, noise_level=0.05):
    #Primary pulse plus 1-3 sub-pulses: hadronic cascade.
    center_main = np.random.uniform(0.25, 0.55)
    amp_main    = np.random.uniform(0.6, 1.0)
    width_main  = np.random.uniform(0.06, 0.12)  # broader than gamma
    waveform    = gaussian_pulse(t, amp_main, center_main, width_main)

    # Add 1-3 sub-pulses at random delays
    n_sub = np.random.randint(1, 4)
    for _ in range(n_sub):
        delay  = np.random.uniform(0.05, 0.35)
        sign   = np.random.choice([-1, 1])
        center_sub = center_main + sign * delay
        amp_sub    = np.random.uniform(0.1, 0.4) * amp_main
        width_sub  = np.random.uniform(0.04, 0.10)
        if 0.0 < center_sub < 1.0:
            waveform += gaussian_pulse(t, amp_sub, center_sub, width_sub)

    waveform += np.random.normal(0, noise_level, size=len(t)).astype(np.float32)
    return waveform.astype(np.float32)


def generate_waveform_dataset(n_gamma=2000, n_hadron=2000):
    """
    Generate a labeled waveform dataset.

    Returns
    -------
    waveforms : (N, 1, N_SAMPLES) float32
    labels    : (N,) int64 — 0 = gamma, 1 = hadron
    """
    gamma_waves  = np.stack([simulate_gamma(noise_level=0)  for _ in range(n_gamma)])
    hadron_waves = np.stack([simulate_hadron(noise_level=0) for _ in range(n_hadron)])

    waveforms = np.concatenate([gamma_waves, hadron_waves], axis=0)
    labels    = np.array([0] * n_gamma + [1] * n_hadron, dtype=np.int64)

    # Normalize to zero mean, unit std across the dataset
    mean = waveforms.mean()
    std  = waveforms.std()
    waveforms = (waveforms - mean) / (std + 1e-8)

    # Add channel dimension: (N, 1, T)
    return waveforms[:, np.newaxis, :].astype(np.float32), labels


waveforms, labels = generate_waveform_dataset(n_gamma=2000, n_hadron=2000)
CLASS_NAMES = ['Gamma', 'Hadron']
print(f'Waveform dataset: {waveforms.shape}, labels: {labels.shape}')

### Visualize synthetic training waveforms

You should see the characteristic structures of the two classes of waveforms. This is where setting `noise_level=0` can be most helpful.

In [ ]:
# Visualize sample waveforms
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for row, (label, name) in enumerate([(0, 'Gamma'), (1, 'Hadron')]):
    idxs = np.where(labels == label)[0][:4]
    for col, idx in enumerate(idxs):
        axes[row, col].plot(T_AXIS, waveforms[idx, 0], lw=1.5)
        axes[row, col].set_title(f'{name} #{col+1}', fontsize=10)
        axes[row, col].set_xlabel('Time')
        if col == 0:
            axes[row, col].set_ylabel('Amplitude')
plt.suptitle('Sample Cherenkov waveforms', fontsize=13)
plt.tight_layout()
plt.show()

## Loading and batching datasets

As usual, we wrap the payload and labels together.

We reserve 15% of the sample for validation.

In [ ]:
class WaveformDataset(Dataset):
    def __init__(self, waveforms, labels):
        self.waveforms = torch.tensor(waveforms, dtype=torch.float32)
        self.labels    = torch.tensor(labels,    dtype=torch.long)

    def __len__(self):
        return len(self.waveforms)

    def __getitem__(self, idx):
        return self.waveforms[idx], self.labels[idx]


dataset = WaveformDataset(waveforms, labels)
n_train = int(0.85 * len(dataset))
n_val   = len(dataset) - n_train
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

### Diffusion parameters

We schedule the time-dependence of the diffusion parameters according to a more complex cosine schedule.

This defines $\bar\alpha$ and $\beta$ for each step in time.
(You will see an exercise later inviting you to change this to a linear schedule.)

In [ ]:
class CosineNoiseSchedule:
    """
    Cosine noise schedule for the diffusion process
    Pre-computes all quantities needed for forward process and training.
    """

    def __init__(self, T=300, s=0.008, device=DEVICE):
        self.T = T
        t      = torch.linspace(0, T, T + 1, device=device)
        f      = torch.cos(((t / T + s) / (1 + s)) * np.pi / 2) ** 2
        alpha_bar         = f / f[0]
        self.alpha_bar    = alpha_bar[1:]              # (T,)
        self.alpha        = self.alpha_bar / torch.cat(
                                [torch.ones(1, device=device), self.alpha_bar[:-1]])
        self.beta         = 1.0 - self.alpha           # (T,)
        self.beta_tilde   = (1.0 - torch.cat(
                                [torch.ones(1, device=device), self.alpha_bar[:-1]])
                             ) / (1.0 - self.alpha_bar) * self.beta
        self.sqrt_alpha_bar       = self.alpha_bar.sqrt()
        self.sqrt_one_minus_alpha_bar = (1.0 - self.alpha_bar).sqrt()

    def q_sample(self, x0, t_idx, eps=None):
        """
        Forward process: sample x_t from x_0 in one step.
        x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps

        Parameters
        ----------
        x0    : (B, 1, T) clean waveforms
        t_idx : (B,) integer timestep indices in [0, T-1]
        eps   : optional pre-sampled noise; if None, sampled here
        """
        if eps is None:
            eps = torch.randn_like(x0)
        a  = self.sqrt_alpha_bar[t_idx].view(-1, 1, 1)
        b  = self.sqrt_one_minus_alpha_bar[t_idx].view(-1, 1, 1)
        return a * x0 + b * eps, eps


schedule = CosineNoiseSchedule(T=300)

# Visualize the schedule
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t_vals = np.arange(1, schedule.T + 1)
axes[0].plot(t_vals, schedule.alpha_bar.cpu().numpy())
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel(r'$\bar{\alpha}_t$')
axes[0].set_title('Signal retention vs timestep')

axes[1].plot(t_vals, schedule.beta.cpu().numpy())
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel(r'$\beta_t$')
axes[1].set_title('Noise schedule $\\beta_t$')
plt.tight_layout()
plt.show()

## Architecture

We will use the U-net architecture as introduced in lecture.

- Encoder: 3 downsampling blocks
- Bottleneck: 1D Conv block
- Decoder: 3 upsampling blocks with skip connections

We did not talk about Feature-wise Linear Modulation (FiLM) in lecture, but the basic idea is to allow the network to behave differently as a function of time and class label.

For example:
- At large t (when there is heavy noise), the network should denoise aggressively and focus on coarse structure. At small t (light noise), it should make fine corrections.
- Gamma waveforms are narrow and single-peaked; hadron waveforms are broad and multi-peaked. The network uses this information to steer the denoising trajectory. This is the advantage of training the denoiser on a target dataset.

Just as we defined positional encoding for the attention-based transformer, we embed some time-dependent coding so that we can track where we are in the diffusion process. This does not violate the idea that we are learning each step in the denoising process one-by-one.

In [ ]:
def sinusoidal_embedding(t, dim=64):
    """
    Sinusoidal timestep embedding, analogous to transformer positional encoding.

    Parameters
    ----------
    t   : (B,) integer timesteps
    dim : embedding dimension (must be even)
    """
    assert dim % 2 == 0
    half  = dim // 2
    freqs = torch.exp(
        -np.log(10000) * torch.arange(half, device=t.device).float() / (half - 1)
    )
    args  = t.float().unsqueeze(1) * freqs.unsqueeze(0)  # (B, half)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)  # (B, dim)


class ResBlock1D(nn.Module):
    """
    1D residual block with FiLM conditioning on (time + class) embedding.
    FiLM: y = gamma(emb) * x + beta(emb)
    """

    def __init__(self, in_ch, out_ch, emb_dim):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch,  out_ch, 3, padding=1)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.act   = nn.SiLU()
        # FiLM: predict scale and shift from embedding
        self.film  = nn.Linear(emb_dim, 2 * out_ch)
        # Residual projection if channel dims differ
        self.skip  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, emb):
        film_out = self.film(emb).unsqueeze(-1)          # (B, 2*out_ch, 1)
        gamma, beta = film_out.chunk(2, dim=1)
        h = self.act(self.norm1(self.conv1(x)))
        h = (1 + gamma) * h + beta                       # FiLM modulation
        h = self.act(self.norm2(self.conv2(h)))
        return h + self.skip(x)


class UNet1D(nn.Module):
    """
    Lightweight 1D U-Net for waveform denoising.
    Input/output shape: (B, 1, N_SAMPLES)
    Conditioned on timestep t and class label c in {0, 1, 2},
    where c=2 is the null (unconditional) token used in classifier-free guidance.
    """

    N_CLASSES = 3  # 0=gamma, 1=hadron, 2=null (unconditional)

    def __init__(self, in_ch=1, base_ch=32, emb_dim=64):
        super().__init__()
        self.emb_dim = emb_dim

        # Time embedding MLP
        self.time_mlp = nn.Sequential(
            nn.Linear(emb_dim, emb_dim * 2),
            nn.SiLU(),
            nn.Linear(emb_dim * 2, emb_dim),
        )
        # Class embedding (includes null token)
        self.class_emb = nn.Embedding(self.N_CLASSES, emb_dim)

        # ── Encoder ──
        self.enc1 = ResBlock1D(in_ch,      base_ch,     emb_dim)
        self.enc2 = ResBlock1D(base_ch,    base_ch * 2, emb_dim)
        self.enc3 = ResBlock1D(base_ch*2,  base_ch * 4, emb_dim)
        self.down  = nn.MaxPool1d(2)

        # ── Bottleneck ──
        self.bot   = ResBlock1D(base_ch * 4, base_ch * 4, emb_dim)

        # ── Decoder ──
        self.up    = nn.Upsample(scale_factor=2, mode='linear', align_corners=False)
        self.dec3  = ResBlock1D(base_ch*4 + base_ch*4, base_ch * 2, emb_dim)
        self.dec2  = ResBlock1D(base_ch*2 + base_ch*2, base_ch,     emb_dim)
        self.dec1  = ResBlock1D(base_ch   + base_ch,   base_ch,     emb_dim)

        self.out   = nn.Conv1d(base_ch, in_ch, 1)

    def forward(self, x, t, c):
        """
        Parameters
        ----------
        x : (B, 1, N_SAMPLES) noisy waveform
        t : (B,) integer timesteps
        c : (B,) integer class labels in {0, 1, 2}
        """
        # Build conditioning embedding
        t_emb = self.time_mlp(sinusoidal_embedding(t, self.emb_dim))
        c_emb = self.class_emb(c)
        emb   = t_emb + c_emb                     # (B, emb_dim)

        # ── Encoder ──
        e1 = self.enc1(x,            emb)          # (B, base_ch,   T)
        e2 = self.enc2(self.down(e1), emb)         # (B, base_ch*2, T/2)
        e3 = self.enc3(self.down(e2), emb)         # (B, base_ch*4, T/4)

        # ── Bottleneck ──
        b  = self.bot(self.down(e3), emb)          # (B, base_ch*4, T/8)

        # ── Decoder with skip connections ──
        d3 = self.dec3(torch.cat([self.up(b),  e3], dim=1), emb)
        d2 = self.dec2(torch.cat([self.up(d3), e2], dim=1), emb)
        d1 = self.dec1(torch.cat([self.up(d2), e1], dim=1), emb)

        return self.out(d1)                        # (B, 1, N_SAMPLES)


net = UNet1D(in_ch=1, base_ch=32, emb_dim=64).to(DEVICE)
n_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f'U-Net parameters: {n_params:,}')

## Training the model

If we have trained the model conditionally using the class labels (FiLM), we have to make sure that information is not used during inference.

During training, we randomly drop the class label with probability
$p_{\text{uncond}} = 0.15$.
This teaches the network both conditional and unconditional denoising
simultaneously, which is needed for classifier-free guidance at
inference time.

In [ ]:
def train_diffusion(net, schedule, train_loader, val_loader,
                    n_epochs=40, lr=3e-4, p_uncond=0.15):
    """
    Train the 1D diffusion model with classifier-free guidance dropout.

    Parameters
    ----------
    p_uncond : probability of replacing the class label with the null token
    """
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    NULL_TOKEN = UNet1D.N_CLASSES - 1  # index 2

    history = {'train_loss': [], 'val_loss': []}

    for epoch in tqdm(range(n_epochs), desc='Training'):
        # ── Training ──
        net.train()
        train_loss_sum = 0.0
        for x0, c in train_loader:
            x0 = x0.to(DEVICE)
            c  = c.to(DEVICE)

            # Randomly drop class labels for classifier-free guidance
            mask = torch.rand(len(c), device=DEVICE) < p_uncond
            c_in = c.clone()
            c_in[mask] = NULL_TOKEN

            # Sample random timesteps
            t_idx = torch.randint(0, schedule.T, (len(x0),), device=DEVICE)

            # Forward process
            xt, eps = schedule.q_sample(x0, t_idx)

            # Predict noise and compute MSE loss
            eps_pred = net(xt, t_idx, c_in)
            loss = F.mse_loss(eps_pred, eps)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
            train_loss_sum += loss.item()

        # ── Validation ──
        net.eval()
        val_loss_sum = 0.0
        with torch.no_grad():
            for x0, c in val_loader:
                x0    = x0.to(DEVICE)
                c     = c.to(DEVICE)
                t_idx = torch.randint(0, schedule.T, (len(x0),), device=DEVICE)
                xt, eps = schedule.q_sample(x0, t_idx)
                eps_pred = net(xt, t_idx, c)
                val_loss_sum += F.mse_loss(eps_pred, eps).item()

        scheduler.step()
        history['train_loss'].append(train_loss_sum / len(train_loader))
        history['val_loss'].append(val_loss_sum   / len(val_loader))

    return history


history = train_diffusion(net, schedule, train_loader, val_loader,
                          n_epochs=40, lr=3e-4)

### Visualize training history

We always do this, just to make sure the model makes progress!

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'],   label='Val')
plt.xlabel('Epoch')
plt.ylabel('MSE loss')
plt.title('Diffusion model training loss')
plt.legend()
plt.tight_layout()
plt.show()

## Generate new waveforms via DDPM

We use the Denoising Diffusion Probabilistic Model, dropping the class labels altogether.

We implemented the guided noise estimate:

$$\hat{\varepsilon}_\theta(x_t, t, c)
    = (1 + w)\,\varepsilon_\theta(x_t, t, c)
    - w\,\varepsilon_\theta(x_t, t, \emptyset)$$

where $w \geq 0$ is the `guidance scale`. (Here $\epsilon \sim N(0,\mathbf{I})$ is the Gaussian noise injected into the decoder.)

Essentially,
- $w = 0$: standard conditional sampling
- $w > 0$: amplifies class-specific features at the cost of diversity

In [ ]:
@torch.no_grad()
def ddpm_sample(net, schedule, n_samples=8, class_label=0, guidance_scale=0.0):
    """
    Generate waveforms by running the full DDPM reverse chain.

    Parameters
    ----------
    class_label    : 0 = gamma, 1 = hadron
    guidance_scale : w in the CFG formula; 0 = no guidance

    Returns
    -------
    x0 : (n_samples, 1, N_SAMPLES) generated waveforms
    """
    net.eval()
    NULL_TOKEN = UNet1D.N_CLASSES - 1
    T = schedule.T

    x = torch.randn(n_samples, 1, N_SAMPLES, device=DEVICE)
    c      = torch.full((n_samples,), class_label, dtype=torch.long, device=DEVICE)
    c_null = torch.full((n_samples,), NULL_TOKEN,  dtype=torch.long, device=DEVICE)

    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, dtype=torch.long, device=DEVICE)

        eps_cond = net(x, t_batch, c)
        if guidance_scale > 0.0:
            eps_uncond = net(x, t_batch, c_null)
            eps = (1 + guidance_scale) * eps_cond - guidance_scale * eps_uncond
        else:
            eps = eps_cond

        # Compute posterior mean (DDPM reverse step)
        alpha_t     = schedule.alpha[t]
        alpha_bar_t = schedule.alpha_bar[t]
        beta_t      = schedule.beta[t]
        beta_tilde  = schedule.beta_tilde[t]
        sqrt_recip_alpha = (1.0 / alpha_t.sqrt())

        mu = sqrt_recip_alpha * (
            x - (beta_t / (1.0 - alpha_bar_t).sqrt()) * eps
        )

        if t > 0:
            z = torch.randn_like(x)
            x = mu + beta_tilde.sqrt() * z
        else:
            x = mu  # No noise at the final step

        # Clip x to prevent amplitude explosion
        x = torch.clamp(x, -5.0, 5.0)

    return x.cpu()

## Visualization of generated waveforms

It's the moment of truth! It took a lot to get to this point, highlighting that diffusion models still take some expertise to handle.

In [ ]:
# Compare generated waveforms to real ones
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

real_gamma  = waveforms[labels == 0][:16, 0, :]
real_hadron = waveforms[labels == 1][:16, 0, :]
gen_gamma   = gamma_samples[:, 0, :].numpy()
gen_hadron  = hadron_samples[:, 0, :].numpy()

for data, ax, title in [
    (real_gamma,  axes[0, 0], 'Real gamma waveforms'),
    (gen_gamma,   axes[0, 1], 'Generated gamma waveforms'),
    (real_hadron, axes[1, 0], 'Real hadron waveforms'),
    (gen_hadron,  axes[1, 1], 'Generated hadron waveforms'),
]:
    mean = data.mean(axis=0)
    std  = data.std(axis=0)
    ax.fill_between(T_AXIS, mean - std, mean + std, alpha=0.3)
    ax.plot(T_AXIS, mean, lw=2, label='Mean')
    for w in data[:5]:
        ax.plot(T_AXIS, w, lw=0.5, alpha=0.4, color='gray')
    ax.set_title(title)
    ax.set_xlabel('Time')
    ax.set_ylabel('Amplitude')

plt.suptitle('Real vs generated waveforms (guidance scale w=2)', fontsize=13)
plt.tight_layout()
plt.show()

Discussion and experiments with the diffusion model

- Repeat the experiments with a different `noise_level`. How do the results change, if at all?

- As you increase the `guidance_scale` $w$, the generated waveforms become more typical for their class but less diverse.
   Explain this in terms of the score function: how do large values for $w$ affect the
   effective score function $\nabla_x \log p_t(x|c)$?

- Replace the cosine schedule function with a linear schedule that drops $\beta_t$ linearly from $10^{-4}$ to $0.02$. Does training converge more quickly or more slowly? How does sample quality change?